In [ ]:
import pandas as pd
import numpy as np

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
import sqlite3


con = sqlite3.connect('../db.sqlite3')
df_orders = pd.read_sql("SELECT * FROM restaurant_order", con)
df_products = pd.read_sql("SELECT * FROM restaurant_product", con)
df_items = pd.read_sql("SELECT * FROM restaurant_orderitem", con)

con.close()

full_df = pd.merge(df_items, df_orders, left_on='order_id', right_on='id', suffixes=('_item', '_order'))
full_df = pd.merge(full_df, df_products, left_on='product_id', right_on='id', suffixes=('', '_prod'))

print(f"Фінальна кількість рядків: {len(full_df)}")

if len(full_df) == 74818:
    print("Успіх! Кількість рядків вірна.")
    full_df.to_csv('combined_data.csv', index=False)
    df = pd.read_csv('combined_data.csv')
else:
    print("Кількість все ще не співпадає. Перевірте назви колонок:")
    print("Items cols:", df_items.columns.tolist())

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
import matplotlib.pyplot as plt

top_10 = df.groupby('name')['quantity'].sum().nlargest(10)

def make_autopct(values):
    def my_autopct(pct):
        total = sum(values)
        val = int(round(pct * total / 100.0))
        return '{:.1f}% ({:d})'.format(pct, val)
    return my_autopct

plt.figure(figsize=(10, 8))

top_10.plot(
    kind='pie',
    autopct=make_autopct(top_10),
    startangle=0,
    title="Top 10 positions in menu by quantity"
)

plt.ylabel("Quantity")
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df['Item Price'] = df['price'] * df['quantity']

top_10_revenue = df.groupby('name')['Item Price'].sum().nlargest(10)

plt.figure(figsize=(10, 8))

top_10_revenue.plot(
    kind='pie',
    autopct=make_autopct(top_10_revenue),
    startangle=0,
    title="Top 10 products by revenue (Item Price)"
)

plt.ylabel("Revenue")
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df['Order Datetime'] = pd.to_datetime(df['datetime'])
df['Order Hour'] = df['Order Datetime'].dt.hour

hourly_profit = df.groupby('Order Hour')['Item Price'].sum()

plt.figure(figsize=(12, 6))

hourly_profit.plot(
    kind='bar',
    color='#4085b0',
    width=0.6
)

plt.title("Profit by Order Hour")
plt.xlabel("Order Hour")
plt.xticks(rotation=0)

plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
df['Day Name'] = df['Order Datetime'].dt.day_name()
df['Day Index'] = df['Order Datetime'].dt.dayofweek

daily_revenue = df.groupby(['Day Index', 'Day Name'])['Item Price'].sum()

days = [index[1] for index in daily_revenue.index]
values = daily_revenue.values

plt.figure(figsize=(10, 6))

plt.bar(days, values, color='#4085b0', width=0.6)

plt.title("Total Restaurant Income by Day of the Week")
plt.xlabel("Day of the Week")
plt.ylabel("Total Revenue")

plt.show()